# Giai đoạn 1: Xử lý dữ liệu (Data Preprocessing)


## Giai đoạn 1: Xử lý dữ liệu (Data Preprocessing)

**Dự án:** Phát hiện gian lận thẻ tín dụng (Credit Card Fraud Detection)

**Notebook này thực hiện các công việc:**
1. Tải dữ liệu
2. Làm sạch và xử lý giá trị thiếu
3. Chuẩn hóa dữ liệu (Scaling)
4. Chia tập dữ liệu Train/Test
5. Áp dụng các kỹ thuật cân bằng dữ liệu: SMOTE, ADASYN, Random Undersampling, Class Weights.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Thư viện tiền xử lý và chia dữ liệu
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler

# Thư viện cân bằng dữ liệu
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter

import warnings
warnings.filterwarnings('ignore')



## 1. Tải và Khám phá bộ dữ liệu


### 1. Tải và Khám phá bộ dữ liệu

**Lưu ý:** Bạn cần để file `creditcard.csv` cùng thư mục với notebook này.

**Tải dữ liệu từ file CSV**

In [ ]:
df = pd.read_csv('creditcard.csv')

# Xem 5 dòng đầu tiên
display(df.head())

# Kiểm tra phân phối của nhãn (Class)
print('Phân phối nhãn (0: Bình thường, 1: Gian lận):')
print(df['Class'].value_counts())

sns.countplot(x='Class', data=df)
plt.title('Phân phối Class trước khi cân bằng')
plt.show()



## 2. Làm sạch dữ liệu và Xử lý giá trị thiếu


### 2. Làm sạch dữ liệu và Xử lý giá trị thiếu

**Kiểm tra giá trị thiếu (Missing values)**

In [ ]:
missing_values = df.isnull().sum().max()
print(f'Số lượng giá trị thiếu lớn nhất trong các cột: {missing_values}')

if missing_values > 0:
    df.dropna(inplace=True)
    print('Đã loại bỏ các dòng có chứa giá trị thiếu.')
else:
    print('Không có giá trị thiếu trong bộ dữ liệu.')

# Kiểm tra dữ liệu trùng lặp
duplicates = df.duplicated().sum()
print(f'Số dòng trùng lặp: {duplicates}')
if duplicates > 0:
    df.drop_duplicates(inplace=True)
    print('Đã loại bỏ các dòng trùng lặp.')



## 3. Chuẩn hóa đặc trưng (Scaling)


### 3. Chuẩn hóa đặc trưng (Scaling)

Hầu hết các đặc trưng (V1-V28) đã được PCA và ẩn danh. Tuy nhiên, Time và Amount chưa được chuẩn hóa. Ta sẽ dùng RobustScaler vì nó ít bị ảnh hưởng bởi các ngoại lệ (outliers) - điều rất phổ biến trong các giao dịch gian lận.

**Khởi tạo RobustScaler**

In [ ]:
rob_scaler = RobustScaler()

# Chuẩn hóa cột Amount và Time
df['scaled_amount'] = rob_scaler.fit_transform(df['Amount'].values.reshape(-1,1))
df['scaled_time'] = rob_scaler.fit_transform(df['Time'].values.reshape(-1,1))

# Bỏ cột Time và Amount cũ
df.drop(['Time','Amount'], axis=1, inplace=True)

# Đưa 2 cột mới lên đầu cho dễ nhìn
scaled_amount = df['scaled_amount']
scaled_time = df['scaled_time']
df.drop(['scaled_amount', 'scaled_time'], axis=1, inplace=True)
df.insert(0, 'scaled_amount', scaled_amount)
df.insert(1, 'scaled_time', scaled_time)

display(df.head())



## 4. Chia tập dữ liệu Train / Test


### 4. Chia tập dữ liệu Train / Test

**CỰC KỲ QUAN TRỌNG:** Việc chia tập dữ liệu phải diễn ra TRƯỚC KHI thực hiện cân bằng dữ liệu (SMOTE/ADASYN). Nếu làm ngược lại sẽ gây ra hiện tượng rò rỉ dữ liệu (Data Leakage) khiến kết quả test không chính xác.

In [ ]:
X = df.drop('Class', axis=1)
y = df['Class']

# Stratify=y để đảm bảo tỷ lệ gian lận trong tập Train và Test là như nhau
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print('Kích thước tập Train:', X_train.shape)
print('Kích thước tập Test:', X_test.shape)
print('Phân phối nhãn tập Train:\n', y_train.value_counts())



## 5. Áp dụng 4 kỹ thuật cân bằng dữ liệu


### 5. Áp dụng 4 kỹ thuật cân bằng dữ liệu

Dưới đây, ta sẽ tạo ra 4 phiên bản tập Train khác nhau tương ứng với 4 kỹ thuật, để sau này dùng cho huấn luyện và so sánh xem kỹ thuật nào tốt nhất.

### 5.1. Undersampling (Lấy mẫu giảm)


#### 5.1. Undersampling (Lấy mẫu giảm)

Giảm ngẫu nhiên số lượng mẫu bình thường sao cho bằng với số lượng mẫu gian lận.

**Lệnh thực hiện Undersampling (Cắt giảm dữ liệu)**

In [ ]:
rus = RandomUnderSampler(random_state=42)
X_train_under, y_train_under = rus.fit_resample(X_train, y_train)

# 2. Lệnh vẽ biểu đồ
plt.figure(figsize=(7, 5))
sns.countplot(x=y_train_under)
plt.title('Phân phối Class sau Undersampling (Tập Train)')
plt.xlabel('Class')
plt.ylabel('Số lượng')
plt.show()

print('Phân phối nhãn sau Undersampling:', Counter(y_train_under))



### 5.2. SMOTE (Sinh mẫu dữ liệu thiểu số)


#### 5.2. SMOTE (Sinh mẫu dữ liệu thiểu số)

Tạo ra các mẫu gian lận giả dựa trên các mẫu có sẵn.

**Visualizing Class Distribution Before and After SMOTE**

**Lệnh tạo dữ liệu giả bằng SMOTE**

In [ ]:
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# 2. Lệnh vẽ biểu đồ
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before SMOTE
sns.countplot(x=y_train, ax=axes[0])
axes[0].set_title('Phân phối Class trước SMOTE (Tập Train)')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Số lượng')

# After SMOTE
sns.countplot(x=y_train_smote, ax=axes[1])
axes[1].set_title('Phân phối Class sau SMOTE (Tập Train)')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Số lượng')

plt.tight_layout()
plt.show()

print('Phân phối nhãn trước SMOTE:', Counter(y_train))
print('Phân phối nhãn sau SMOTE:', Counter(y_train_smote))



### 5.3. ADASYN


#### 5.3. ADASYN

In [ ]:
adasyn = ADASYN(random_state=42)
X_train_adasyn, y_train_adasyn = adasyn.fit_resample(X_train, y_train)
print('Phân phối nhãn sau ADASYN:', Counter(y_train_adasyn))



### 5.4. Class Weights (Trọng số lớp)


#### 5.4. Class Weights (Trọng số lớp)

In [ ]:
from sklearn.utils import class_weight
weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
print("Class weights:", weights)



## 6. Lưu dữ liệu đã xử lý để dùng cho Giai đoạn 2 (Modeling)


### 6. Lưu dữ liệu đã xử lý để dùng cho Giai đoạn 2 (Modeling)

Lưu các biến X_train_smote, y_train_smote... ra file để nhóm chạy Model có thể dùng trực tiếp.

In [ ]:
import pickle

# Lưu tập Train SMOTE làm ví dụ
with open('train_smote.pkl', 'wb') as f:
    pickle.dump((X_train_smote, y_train_smote), f)

with open('test_data.pkl', 'wb') as f:
    pickle.dump((X_test, y_test), f)

print('Hoàn tất Giai đoạn 1: Tiền xử lý dữ liệu!')

!pip install catboost lightgbm shap -q




# Giai đoạn 2: Xây dựng và Huấn luyện Mô hình (Modeling)


## Giai đoạn 2: Xây dựng và Huấn luyện Mô hình (Modeling)

==============================================================================

In [ ]:
import pickle
from xgboost import XGBClassifier
from sklearn.ensemble import ExtraTreesClassifier

# --- BẮT ĐẦU BỔ SUNG THƯ VIỆN TỪ FILE CỦA BẠN ---
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
# --- KẾT THÚC BỔ SUNG ---

# Đọc dữ liệu train
with open('train_smote.pkl', 'rb') as f:
    X_train, y_train = pickle.load(f)

print("Đang huấn luyện mô hình...")

# XGBoost
xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)
xgb_model.fit(X_train, y_train)

# ExtraTrees
et_model = ExtraTreesClassifier(
    n_estimators=100,
    random_state=42
)
et_model.fit(X_train, y_train)


# --- BẮT ĐẦU BỔ SUNG MÔ HÌNH TỪ FILE CỦA BẠN ---
# LightGBM
lgbm_model = LGBMClassifier(n_estimators=100, learning_rate=0.1, random_state=42, verbose=-1)
lgbm_model.fit(X_train, y_train)

# CatBoost
cat_model = CatBoostClassifier(iterations=100, learning_rate=0.1, verbose=0, random_state=42)
cat_model.fit(X_train, y_train)

# XGBoost ClassWeight 
# Lưu ý: class_weight lấy từ biến 'weights' tính ở Phase 1 của team
xgb_weight_model = XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, scale_pos_weight=weights[1], random_state=42)
xgb_weight_model.fit(X_train, y_train)
# --- KẾT THÚC BỔ SUNG ---

# Lưu các model (Team's structure)
models = {
    "XGBoost": xgb_model,
    "ExtraTrees": et_model,
    "LightGBM": lgbm_model,             # (Bổ sung của bạn)
    "CatBoost": cat_model,              # (Bổ sung của bạn)
    "XGBoost_ClassWeight": xgb_weight_model # (Bổ sung của bạn)
}

with open('trained_models_smote.pkl', 'wb') as f:
    pickle.dump(models, f)

print("Đã tạo file trained_models_smote.pkl")




# Giai đoạn 3: Đánh Giá (Evaluation)


## Giai đoạn 3: Đánh Giá (Evaluation)

==============================================================================

In [ ]:
import pickle
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

print("ĐANG ĐÁNH GIÁ MÔ HÌNH...\n")

# Load test data
with open('test_data.pkl', 'rb') as f:
    X_test, y_test = pickle.load(f)

# Load trained models
with open('trained_models_smote.pkl', 'rb') as f:
    models = pickle.load(f)

with open('Bang_Diem_AI.txt', 'w', encoding='utf-8') as f_out:

    for name, model in models.items():

        print(f"\n===== {name} =====")

        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:,1]

        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        roc_auc = roc_auc_score(y_test, y_prob)
        pr_auc = average_precision_score(y_test, y_prob)

        print("Precision:", precision)
        print("Recall:", recall)
        print("F1 Score:", f1)
        print("ROC-AUC:", roc_auc)
        print("PR-AUC:", pr_auc)

        f_out.write(f"\n===== {name} =====\n")
        f_out.write(f"Precision: {precision:.4f}\n")
        f_out.write(f"Recall: {recall:.4f}\n")
        f_out.write(f"F1 Score: {f1:.4f}\n")
        f_out.write(f"ROC-AUC: {roc_auc:.4f}\n")
        f_out.write(f"PR-AUC: {pr_auc:.4f}\n\n")

        f_out.write(classification_report(y_test, y_pred))
        f_out.write("\n" + "="*60 + "\n")

        cm = confusion_matrix(y_test, y_pred)

        plt.figure(figsize=(6,4))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
        plt.title(f'Confusion Matrix - {name}')
        plt.savefig(f'Confusion_Matrix_{name}.png')
        plt.close()

print("\nHOÀN THÀNH ĐÁNH GIÁ")




# Giai đoạn 4: Giải Thích Mô hình (ExAI)


## Giai đoạn 4: Giải Thích Mô hình (ExAI)

## 4.1. Cấu hình SHAP & Xác định mô hình tốt nhất


### 4.1. Cấu hình SHAP & Xác định mô hình tốt nhất

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score
import shap

print("--- ĐANG KHỞI TẠO SHAP & TÌM BEST MODEL ---")

# 1. Tải tập dữ liệu Test và các mô hình đã huấn luyện
with open('test_data.pkl', 'rb') as f:
    X_test, y_test = pickle.load(f)

with open('trained_models_smote.pkl', 'rb') as f:
    models = pickle.load(f)

# 2. Tự động tìm mô hình xuất sắc nhất dựa trên PR-AUC
best_model_name = None
best_model_obj = None
best_pr_auc = -1

for name, model in models.items():
    y_prob = model.predict_proba(X_test)[:, 1]
    pr_auc = average_precision_score(y_test, y_prob)
    if pr_auc > best_pr_auc:
        best_pr_auc = pr_auc
        best_model_name = name
        best_model_obj = model

print(f"🏆 Mô hình xuất sắc nhất được chọn: {best_model_name} (PR-AUC = {best_pr_auc:.4f})")

# 3. Trích xuất mẫu dữ liệu nhỏ để chạy SHAP nhanh, không treo RAM
X_exai_sample = X_test.sample(100, random_state=42)

# 4. Khởi tạo bộ giải thích cây (TreeExplainer) và tính toán shap_values
explainer = shap.TreeExplainer(best_model_obj)
shap_values_raw = explainer.shap_values(X_exai_sample)

# Đồng bộ hóa cấu trúc đầu ra giữa các dòng thuật toán (XGBoost / ExtraTrees)
if isinstance(shap_values_raw, list):
    final_shap_values = shap_values_raw[1] if len(shap_values_raw) > 1 else shap_values_raw[0]
elif len(shap_values_raw.shape) == 3:
    final_shap_values = shap_values_raw[:, :, 1]
else:
    final_shap_values = shap_values_raw

print("✅ Đã tính toán xong giá trị SHAP. Sẵn sàng vẽ biểu đồ ở các ô tiếp theo!")



## 4.2. Biểu đồ Feature Importance (Mức độ quan trọng của đặc trưng)


### 4.2. Biểu đồ Feature Importance (Mức độ quan trọng của đặc trưng)

In [ ]:
print(f"[Đang vẽ] Feature Importance cho mô hình: {best_model_name}")

plt.figure(figsize=(10, 5))
# Tăng fontsize=16 giúp tiêu đề biểu đồ to và rõ ràng
plt.title(f"SHAP Feature Importance - {best_model_name}", fontsize=16, fontweight='bold', pad=15)

shap.summary_plot(final_shap_values, X_exai_sample, plot_type="bar", show=False)

plt.tight_layout()
plt.savefig(f'SHAP_Feature_Importance_{best_model_name}.png', dpi=300)
plt.show()
plt.close()




# Giai đoạn 5: Chuẩn bị Web Deploy


## Giai đoạn 5: Chuẩn bị Web Deploy

In [ ]:
import joblib

print("\n--- CHUẨN BỊ XUẤT MÔ HÌNH CHO STREAMLIT WEB APP ---")

# Xuất mô hình tốt nhất
model_filename = 'fraud_model.pkl'
joblib.dump(best_model_obj, model_filename)
print(f"✅ Đã lưu mô hình tốt nhất ({best_model_name}) thành: {model_filename}")

# Xuất Scaler
try:
    scaler_filename = 'scaler.pkl'
    joblib.dump(rob_scaler, scaler_filename) 
    print(f"✅ Đã lưu bộ chuẩn hóa RobustScaler thành: {scaler_filename}")
except NameError:
    print("⚠️ Không tìm thấy biến rob_scaler trong bộ nhớ.")

print("\n🎉 TẤT CẢ ĐÃ SẴN SÀNG ĐỂ DEPLOY!")